# Workshop 3 — Governance & Storytelling: Data Quality Findings and Gold Layer Analysis

**Proyecto:** Music Artists & Albums Public Perception  
**Curso:** Programación para Análisis de Datos — Semestre 2026-I  
**Universidad Distrital Francisco José de Caldas**

Este notebook documenta:
1. **Data Quality Findings** — estadísticas descriptivas, null rates, outliers y distribuciones de longitud de texto sobre la capa Silver
2. **Governance KPIs** — lectura del Parquet gold `governance_*.parquet` con justificación de cada KPI
3. **Storytelling Aggregations** — lectura del Parquet gold `storytelling_*.parquet` con todas las agregaciones para el dashboard
4. **Dashboard Narrative** — qué historia cuentan los datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os, glob

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

# ── Rutas ──────────────────────────────────────────────────────────────────
NB_DIR    = os.path.dirname(os.path.abspath('__file__'))  # notebooks/
ROOT_DIR  = os.path.dirname(NB_DIR)
GOLD_DIR  = os.path.join(ROOT_DIR, 'datalake_gold')

# CSV exports (producidos localmente desde los Parquet silver)
CSV_REDDIT   = os.path.join(NB_DIR, 'export_reddit.csv')
CSV_ARTISTS  = os.path.join(NB_DIR, 'export_artists.csv')
CSV_TRACKS   = os.path.join(NB_DIR, 'export_tracks.csv')
CSV_GOV      = os.path.join(NB_DIR, 'export_governance.csv')
CSV_STORY    = os.path.join(NB_DIR, 'export_storytelling.csv')

print('Directorio raíz:', ROOT_DIR)
print('Directorio gold:', GOLD_DIR)

---
## PARTE 1 — Calidad de Datos en la Capa Silver

Se cargan los datos silver exportados como CSV. Se analiza cada fuente de forma independiente.

In [ ]:
# ── Cargar datos Silver ────────────────────────────────────────────────────
df_reddit  = pd.read_csv(CSV_REDDIT)
df_artists = pd.read_csv(CSV_ARTISTS)
df_tracks  = pd.read_csv(CSV_TRACKS)

print(f'Reddit silver:  {len(df_reddit):>5} filas × {df_reddit.shape[1]} columnas')
print(f'Artists silver: {len(df_artists):>5} filas × {df_artists.shape[1]} columnas')
print(f'Tracks silver:  {len(df_tracks):>5} filas × {df_tracks.shape[1]} columnas')

### 1.1 Estadísticas Descriptivas — Reddit

In [ ]:
num_cols_reddit = ['score', 'word_count', 'confidence', 'extract_confidence', 'score_capped', 'word_count_capped']
df_reddit[num_cols_reddit].describe().round(4)

**Hallazgos clave Reddit:**
- `score` tiene alta varianza (posts van de 0 a 607 puntos) → justifica el capping IQR aplicado en silver.
- `word_count` va de 1 a valores extremos → el capping reduce el rango efectivo sin eliminar filas.
- `confidence` media ≈ 0.17 → la mayoría de comentarios son opiniones libres, no recomendaciones estructuradas con patrón artista/canción.
- `extract_confidence` ≈ 0.0 → muy pocos comentarios tienen el formato `Artist - Song` o similar, lo que refleja la naturaleza orgánica de la discusión.

In [ ]:
# ── Estadísticas descriptivas — Last.fm Artists ───────────────────────────
print('=== Last.fm Top Artists ===')
display(df_artists[['playcount','listeners']].describe().round(0))

print('\n=== Last.fm Top Tracks ===')
display(df_tracks[['playcount','listeners','duration_sec']].describe().round(0))

**Hallazgos clave Last.fm:**
- `playcount` de artistas varía de millones a miles de millones → distribución muy sesgada, outlier rate esperado alto.
- `listeners` de tracks: rango amplio pero menos extremo que playcount → indica artistas con seguidores fieles vs. hits puntuales.
- `duration_sec`: la mayoría entre 150 y 300 s (2.5–5 min), coherente con canciones comerciales.

### 1.2 Null Rate por Campo

In [ ]:
def null_rate_table(df, source_name):
    total = len(df)
    nulls = df.isnull().sum()
    unknown_mask = df.apply(lambda c: (c == 'unknown').sum() if c.dtype == object else 0)
    result = pd.DataFrame({
        'null_count':    nulls,
        'unknown_count': unknown_mask,
        'null_rate_%':   (nulls / total * 100).round(2),
        'unknown_rate_%':(unknown_mask / total * 100).round(2),
    })
    result.index.name = 'field'
    print(f'\n── {source_name} ({total} registros) ──')
    display(result[result['null_count'] + result['unknown_count'] > 0]
            .sort_values('null_rate_%', ascending=False))
    return result

nr_reddit   = null_rate_table(df_reddit,  'Reddit')
nr_artists  = null_rate_table(df_artists, 'Last.fm Artists')
nr_tracks   = null_rate_table(df_tracks,  'Last.fm Tracks')

In [ ]:
# ── Visualización null rates Reddit ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (df, title) in zip(axes, [
    (df_reddit[['artist','song','pattern_type']], 'Reddit — Campos Opcionales'),
    (df_artists[['mbid']], 'Last.fm Artists'),
    (df_tracks[['mbid','artist_mbid']], 'Last.fm Tracks'),
]):
    rates = (df == 'unknown').mean() * 100
    if rates.empty or rates.sum() == 0:
        ax.text(0.5, 0.5, 'Sin campos opcionales\ncon valores desconocidos',
                ha='center', va='center', transform=ax.transAxes)
    else:
        bars = ax.barh(rates.index, rates.values, color='#e07b54', edgecolor='white')
        ax.set_xlabel('% → "unknown"')
        ax.set_xlim(0, 105)
        for bar, val in zip(bars, rates.values):
            ax.text(val + 1, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%', va='center', fontsize=9)
    ax.set_title(title)

plt.suptitle('Tasa de valores opcionales → "unknown" por fuente', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

**Justificación KPI — Null Rate:** Los campos `artist` y `song` en Reddit presentan alta tasa de `"unknown"` porque la mayoría de comentarios son opiniones libres sin el formato estructurado `Artista - Canción`. El campo `mbid` de Last.fm tiene una tasa de `"unknown"` que indica que no todas las entidades están registradas en MusicBrainz. Ambos campos son **opcionales** en el schema, por lo que no afectan la schema compliance rate.

### 1.3 Análisis de Outliers (IQR)

In [ ]:
def iqr_outlier_analysis(series, label):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((series < lower) | (series > upper)).sum()
    return {
        'campo': label, 'Q1': round(q1, 1), 'Q3': round(q3, 1),
        'IQR': round(iqr, 1), 'lower_fence': round(lower, 1),
        'upper_fence': round(upper, 1), 'n_outliers': n_out,
        'outlier_rate_%': round(n_out / len(series) * 100, 2)
    }

outlier_rows = [
    iqr_outlier_analysis(df_reddit['score'],      'reddit.score'),
    iqr_outlier_analysis(df_reddit['word_count'], 'reddit.word_count'),
    iqr_outlier_analysis(df_reddit['confidence'], 'reddit.confidence'),
    iqr_outlier_analysis(df_artists['playcount'], 'artists.playcount'),
    iqr_outlier_analysis(df_artists['listeners'], 'artists.listeners'),
    iqr_outlier_analysis(df_tracks['playcount'],  'tracks.playcount'),
    iqr_outlier_analysis(df_tracks['listeners'],  'tracks.listeners'),
    iqr_outlier_analysis(df_tracks['duration_sec'], 'tracks.duration_sec'),
]

pd.DataFrame(outlier_rows).set_index('campo')

In [ ]:
# ── Boxplots de campos numéricos clave ────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

pairs = [
    (df_reddit['score'],           'Reddit: score'),
    (df_reddit['word_count'],      'Reddit: word_count'),
    (df_reddit['confidence'],      'Reddit: confidence'),
    (df_reddit['extract_confidence'], 'Reddit: extract_confidence'),
    (df_artists['playcount'],      'Artists: playcount'),
    (df_artists['listeners'],      'Artists: listeners'),
    (df_tracks['playcount'],       'Tracks: playcount'),
    (df_tracks['duration_sec'],    'Tracks: duration_sec'),
]

for ax, (series, title) in zip(axes.flat, pairs):
    ax.boxplot(series.dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='#5b9bd5', alpha=0.7),
               medianprops=dict(color='#c00000', linewidth=2),
               flierprops=dict(marker='o', markerfacecolor='#e07b54', markersize=4, alpha=0.5))
    ax.set_title(title, fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1e9:.1f}B' if x >= 1e9 else (f'{x/1e6:.0f}M' if x >= 1e6 else f'{x:.0f}')))
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Distribución de campos numéricos — detección de outliers IQR', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**Hallazgos de Outliers:**
- **`reddit.score`**: Posts virales (score > 500) elevan la media. El capping IQR aplicado en silver preserva la variación normal sin que un post viral distorsione los análisis.
- **`reddit.word_count`**: Comentarios muy largos (>50 palabras) son outliers — generalmente son hilos de recomendaciones extensas. El capping los normaliza sin eliminarlos.
- **`artists.playcount` / `tracks.playcount`**: Distribución extremadamente sesgada a la derecha (artistas como BTS con 4B+ plays). La mayoría de artistas tienen <500M plays.

### 1.4 Distribuciones de Longitud de Texto

In [ ]:
df_reddit['len_raw']   = df_reddit['raw_comment'].str.len()
df_reddit['len_clean'] = df_reddit['clean_comment'].str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reddit: raw vs clean
axes[0].hist(df_reddit['len_raw'].dropna(),   bins=40, alpha=0.6, color='#5b9bd5', label='raw_comment')
axes[0].hist(df_reddit['len_clean'].dropna(), bins=40, alpha=0.6, color='#e07b54', label='clean_comment')
axes[0].set_xlabel('Longitud (caracteres)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Reddit: longitud raw vs clean')
axes[0].legend()

# Reddit: word count
axes[1].hist(df_reddit['word_count'].dropna(), bins=30, color='#70ad47', edgecolor='white')
axes[1].axvline(df_reddit['word_count'].median(), color='red', linestyle='--', label=f'Mediana: {df_reddit["word_count"].median():.0f}')
axes[1].axvline(df_reddit['word_count'].mean(),   color='orange', linestyle='--', label=f'Media: {df_reddit["word_count"].mean():.0f}')
axes[1].set_xlabel('Palabras')
axes[1].set_title('Reddit: distribución word_count')
axes[1].legend(fontsize=8)

# Last.fm: longitud de nombres de artista
df_artists['len_name'] = df_artists['name'].str.len()
axes[2].hist(df_artists['len_name'].dropna(), bins=20, color='#9e69af', edgecolor='white')
axes[2].set_xlabel('Longitud (caracteres)')
axes[2].set_title('Last.fm Artists: longitud de nombre')

plt.suptitle('Distribuciones de longitud de texto por fuente', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Tabla resumen
text_stats = pd.DataFrame({
    'campo': ['reddit.raw_comment', 'reddit.clean_comment', 'reddit.word_count', 'artists.name', 'tracks.name'],
    'mean':  [df_reddit['len_raw'].mean(), df_reddit['len_clean'].mean(), df_reddit['word_count'].mean(),
              df_artists['name'].str.len().mean(), df_tracks['name'].str.len().mean()],
    'median':[df_reddit['len_raw'].median(), df_reddit['len_clean'].median(), df_reddit['word_count'].median(),
              df_artists['name'].str.len().median(), df_tracks['name'].str.len().median()],
    'min':   [df_reddit['len_raw'].min(), df_reddit['len_clean'].min(), df_reddit['word_count'].min(),
              df_artists['name'].str.len().min(), df_tracks['name'].str.len().min()],
    'max':   [df_reddit['len_raw'].max(), df_reddit['len_clean'].max(), df_reddit['word_count'].max(),
              df_artists['name'].str.len().max(), df_tracks['name'].str.len().max()],
}).set_index('campo').round(1)

display(text_stats)

### 1.5 Distribución de Tipos de Comentario y Patrones Musicales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Tipos de comentario
ct = df_reddit['comment_type'].value_counts()
colors = {'recommendation':'#5b9bd5', 'opinion':'#70ad47', 'mixed':'#ffc000', 'other':'#e07b54'}
bar_colors = [colors.get(k, '#aaa') for k in ct.index]
axes[0].barh(ct.index, ct.values, color=bar_colors, edgecolor='white')
for i, (idx, val) in enumerate(ct.items()):
    axes[0].text(val + 1, i, f'{val} ({val/len(df_reddit)*100:.1f}%)', va='center', fontsize=9)
axes[0].set_xlabel('Cantidad de comentarios')
axes[0].set_title('Tipo de comentario')
axes[0].set_xlim(0, ct.max() * 1.25)

# Patrón musical
pattern_counts = df_reddit['pattern_type'].value_counts()
axes[1].bar(pattern_counts.index, pattern_counts.values, color='#5b9bd5', edgecolor='white')
axes[1].set_title('Tipo de patrón musical detectado')
axes[1].set_ylabel('Cantidad')

# Distribución de confidence
for ct_val, color in colors.items():
    subset = df_reddit[df_reddit['comment_type'] == ct_val]['confidence']
    if len(subset) > 0:
        axes[2].hist(subset, bins=20, alpha=0.6, color=color, label=ct_val)
axes[2].set_xlabel('confidence')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribución de confidence por tipo')
axes[2].legend(fontsize=8)

plt.suptitle('Clasificación NLP de comentarios Reddit', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**Hallazgos — Clasificación NLP:**
- **78.5% son opiniones** (`opinion`): los usuarios de r/indieheads y r/hiphopheads escriben en forma discursiva, no estructurada.
- **0% son recomendaciones puras**: la ausencia del patrón `Artista - Canción` en la mayoría de posts `[FRESH ALBUM]` indica que los usuarios comentan sobre el álbum en general, no sobre canciones específicas.
- **Solo 1 mixed**: la intersección entre patrón musical y opinión es casi nula → los pocos comentarios con patrón estructurado no incluyen marcadores de contraste.
- **confidence media baja (≈0.17)**: coherente con la dominancia del tipo `opinion` y `other`, que reciben confidence = 0 al no tener patrón musical.

---
## PARTE 2 — Governance KPIs (Capa Gold)

Se carga el CSV exportado desde `governance_*.parquet`. Para cargar el Parquet directamente usar `pd.read_parquet()`.

In [ ]:
# ── Cargar governance ─────────────────────────────────────────────────────
df_gov = pd.read_csv(CSV_GOV)

# Opcionalmente, cargar desde Parquet gold
gov_parquets = sorted(glob.glob(os.path.join(GOLD_DIR, 'governance_*.parquet')))
if gov_parquets:
    df_gov_pq = pd.read_parquet(gov_parquets[-1])
    print(f'Parquet cargado: {gov_parquets[-1]}')
    print(f'Filas: {len(df_gov_pq)} | Cols: {df_gov_pq.columns.tolist()}')

print(f'\nCSV governance: {len(df_gov)} KPIs')
df_gov.head(8)

In [ ]:
# ── Pivot: KPI por fuente ─────────────────────────────────────────────────
vol = df_gov[df_gov['kpi_type'] == 'volume'][['source','value']].set_index('source').rename(columns={'value':'total_records'})
comp = df_gov[df_gov['kpi_type'] == 'schema_compliance'][['source','value']].set_index('source').rename(columns={'value':'schema_compliance_%'})
days = df_gov[df_gov['kpi_type'] == 'ingestion_days'][['source','value']].set_index('source').rename(columns={'value':'ingestion_days'})

summary_kpi = vol.join(comp).join(days)
summary_kpi['total_records'] = summary_kpi['total_records'].astype(int)
display(summary_kpi)

In [ ]:
# ── Null rates por fuente (mapa de calor) ─────────────────────────────────
null_gov = df_gov[df_gov['kpi_type'] == 'null_rate'].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, source in zip(axes, ['reddit', 'lastfm_artists', 'lastfm_tracks']):
    sub = null_gov[null_gov['source'] == source].sort_values('value', ascending=False)
    colors_bar = ['#c00000' if v > 5 else ('#ffc000' if v > 0 else '#70ad47') for v in sub['value']]
    ax.barh(sub['field_name'], sub['value'], color=colors_bar, edgecolor='white')
    ax.set_xlabel('Null rate (%)')
    ax.set_title(f'{source}')
    ax.axvline(0, color='black', linewidth=0.5)

plt.suptitle('Null Rate (%) por campo y fuente — 🟢 0% · 🟡 >0% · 🔴 >5%', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Outlier rate por campo numérico ───────────────────────────────────────
outlier_gov = df_gov[df_gov['kpi_type'] == 'outlier_rate'].copy()

fig, ax = plt.subplots(figsize=(10, 5))
palette = {'reddit':'#5b9bd5', 'lastfm_artists':'#70ad47', 'lastfm_tracks':'#ffc000'}

for i, (source, grp) in enumerate(outlier_gov.groupby('source')):
    grp = grp.sort_values('value', ascending=False)
    ax.barh([f'{source}.{f}' for f in grp['field_name']], grp['value'],
            color=palette.get(source, '#aaa'), alpha=0.85, edgecolor='white', label=source)

ax.set_xlabel('Outlier rate (%)')
ax.set_title('Outlier Rate (IQR ×1.5) por campo numérico')
ax.legend()
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### 2.1 Tabla de Justificación de KPIs

| KPI | Fuente de observación | Hallazgo que lo justifica | Severidad |
|---|---|---|---|
| **Null rate por campo** | Todos | `artist` y `song` en Reddit tienen >95% `unknown`; `mbid` en Last.fm también alto | Media |
| **Volumen total** | Todos | Reddit: 228 registros; Artists: variable por día (50/ingesta); Tracks: variable | Baja |
| **Schema compliance** | Todos | 100% en todos — el pipeline silver enforcea schema antes de escribir | N/A |
| **Outlier rate (IQR)** | Reddit, Last.fm | `score` Reddit y `playcount` Last.fm tienen outliers significativos | Alta |
| **Text length stats** | Reddit | `raw_comment` ≫ `clean_comment` confirma que la limpieza reduce ruido | Media |
| **Ingestion days** | Last.fm | Múltiples snapshots diarios permiten ver tendencias temporales | Baja |
| **Ingestion frequency compliance** | Last.fm | Expected: 1 run/día · Actual: ~13 snapshots en 2 días → on track | N/A |

---
## PARTE 3 — Storytelling Aggregations (Capa Gold)

In [ ]:
# ── Cargar storytelling ───────────────────────────────────────────────────
df_story = pd.read_csv(CSV_STORY)

story_parquets = sorted(glob.glob(os.path.join(GOLD_DIR, 'storytelling_*.parquet')))
if story_parquets:
    df_story_pq = pd.read_parquet(story_parquets[-1])
    print(f'Parquet cargado: {story_parquets[-1]}')

print(f'Storytelling rows: {len(df_story)}')
print('Metric types:', df_story['metric_type'].unique().tolist())
df_story.head(5)

### 3.1 Distribución de Sentimiento (VADER)

**User Story vinculada:** *«Como analista de un sello discográfico, quiero saber si la comunidad reacciona positiva o negativamente a los nuevos lanzamientos para priorizar campañas de comunicación.»*

In [ ]:
sent = df_story[df_story['metric_type'] == 'sentiment_dist'].copy()
sent_colors = {'positive':'#70ad47', 'negative':'#c00000', 'neutral':'#5b9bd5'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barras
bar_colors = [sent_colors.get(d, '#aaa') for d in sent['dim1']]
bars = axes[0].bar(sent['dim1'], sent['pct'], color=bar_colors, edgecolor='white', width=0.5)
for bar, row in zip(bars, sent.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{row.pct:.1f}%\n(n={row.record_count})', ha='center', fontsize=9)
axes[0].set_ylabel('Porcentaje (%)')
axes[0].set_title('Distribución de sentimiento (VADER)')
axes[0].set_ylim(0, 85)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Torta
wedge_colors = [sent_colors.get(d, '#aaa') for d in sent['dim1']]
axes[1].pie(sent['record_count'], labels=sent['dim1'], colors=wedge_colors,
            autopct='%1.1f%%', startangle=90, pctdistance=0.75,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Distribución de sentimiento')

plt.suptitle('Análisis de sentimiento — Reddit (r/indieheads + r/hiphopheads)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Tabla detalle
display(sent[['dim1','record_count','pct','avg_score']].rename(
    columns={'dim1':'sentiment','record_count':'count','avg_score':'avg_compound_score'}))

**Insight:** El **67.98% de los comentarios son positivos** (score VADER promedio: +0.62), un **17.98% neutros** y solo el **14.04% negativos** (score: -0.52). La comunidad indie y hip-hop reacciona con entusiasmo a los nuevos lanzamientos analizados.

### 3.2 Tendencia de Sentimiento en el Tiempo

**User Story:** *«Como manager de artista, quiero ver si la percepción mejora o empeora a lo largo del tiempo para ajustar la estrategia de lanzamiento.»*

In [ ]:
trend = df_story[df_story['metric_type'] == 'sentiment_trend'].copy()
trend['dim1'] = pd.to_datetime(trend['dim1'])
trend = trend.sort_values('dim1')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trend['dim1'], trend['avg_score'], marker='o', color='#5b9bd5', linewidth=2, markersize=7)
ax.fill_between(trend['dim1'], 0, trend['avg_score'],
                where=trend['avg_score'] >= 0, color='#70ad47', alpha=0.2, label='Positivo')
ax.fill_between(trend['dim1'], 0, trend['avg_score'],
                where=trend['avg_score'] < 0, color='#c00000', alpha=0.2, label='Negativo')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Score VADER promedio')
ax.set_xlabel('Fecha de ingesta')
ax.set_title('Tendencia de sentimiento por fecha (compound score promedio)')
ax.legend()
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

display(trend[['dim1','record_count','avg_score']].rename(
    columns={'dim1':'fecha','record_count':'comentarios','avg_score':'avg_compound'}))

### 3.3 Top Keywords

**User Story:** *«Como analista, quiero identificar qué términos dominan la conversación para entender qué aspectos del lanzamiento generan más reacción.»*

In [ ]:
kw = df_story[df_story['metric_type'] == 'top_keyword'].copy()
kw = kw.sort_values('record_count', ascending=True).tail(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Frecuencia
sentiment_color = ['#70ad47' if s > 0.05 else ('#c00000' if s < -0.05 else '#5b9bd5')
                   for s in kw['avg_score']]
axes[0].barh(kw['dim1'], kw['record_count'], color=sentiment_color, edgecolor='white')
axes[0].set_xlabel('Frecuencia')
axes[0].set_title('Top 20 keywords (color = sentimiento)')
axes[0].text(0.98, 0.02, '🟢 positivo  🔵 neutro  🔴 negativo',
             transform=axes[0].transAxes, ha='right', fontsize=8)

# Sentimiento por keyword
kw_sorted = kw.sort_values('avg_score', ascending=True)
bar_colors2 = ['#70ad47' if s > 0.05 else ('#c00000' if s < -0.05 else '#5b9bd5')
               for s in kw_sorted['avg_score']]
axes[1].barh(kw_sorted['dim1'], kw_sorted['avg_score'], color=bar_colors2, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Score VADER promedio')
axes[1].set_title('Sentimiento promedio asociado a cada keyword')

plt.suptitle('Análisis de keywords — Reddit', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### 3.4 Top Artistas y Tracks — Last.fm

**User Story:** *«Como analista, quiero ver el ranking de popularidad cuantitativa para compararlo con la percepción cualitativa de Reddit.»*

In [ ]:
artists_st = df_story[df_story['metric_type'] == 'top_artist_lastfm'].sort_values('avg_score', ascending=False).head(15)
tracks_st  = df_story[df_story['metric_type'] == 'top_track_lastfm'].sort_values('record_count', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top Artists por listeners
axes[0].barh(artists_st['dim1'][::-1], artists_st['avg_score'][::-1] / 1e6,
             color='#5b9bd5', edgecolor='white')
axes[0].set_xlabel('Listeners únicos (millones)')
axes[0].set_title('Top 15 Artistas — Last.fm (por listeners únicos)')
axes[0].grid(axis='x', linestyle='--', alpha=0.4)

# Top Tracks por playcount
axes[1].barh(
    [f"{r.dim1[:25]}…\n({r.dim2})" if len(r.dim1) > 25 else f"{r.dim1}\n({r.dim2})"
     for r in tracks_st.iloc[::-1].itertuples()],
    tracks_st['record_count'][::-1] / 1e6,
    color='#70ad47', edgecolor='white'
)
axes[1].set_xlabel('Reproducciones (millones)')
axes[1].set_title('Top 15 Tracks — Last.fm (por reproducciones)')
axes[1].grid(axis='x', linestyle='--', alpha=0.4)

plt.suptitle('Rankings de popularidad cuantitativa — Last.fm', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## PARTE 4 — Reporte de Hallazgos de Calidad de Datos

Resumen estructurado para la sección del informe de Workshop 3.

In [ ]:
findings = [
    {'Fuente': 'Reddit', 'Campo': 'artist / song', 'Naturaleza': 'Alta tasa de "unknown"',
     'Frecuencia': '>95% de filas', 'Severidad': 'Media',
     'Estrategia': 'Campos opcionales en schema; pipeline de extracción regex (dash/by/colon) cubre los casos estructurados'},
    {'Fuente': 'Reddit', 'Campo': 'score', 'Naturaleza': 'Outliers extremos (posts virales)',
     'Frecuencia': '~15% de filas fuera IQR×1.5', 'Severidad': 'Alta',
     'Estrategia': 'Capping IQR → score_capped; el campo original score se preserva para trazabilidad'},
    {'Fuente': 'Reddit', 'Campo': 'word_count', 'Naturaleza': 'Comentarios extremadamente largos (outliers)',
     'Frecuencia': '~10% de filas', 'Severidad': 'Media',
     'Estrategia': 'Capping IQR → word_count_capped; comentarios no se truncan, solo se normaliza la métrica'},
    {'Fuente': 'Reddit', 'Campo': 'comment_type', 'Naturaleza': 'Ausencia de recomendaciones estructuradas',
     'Frecuencia': '0% recommendations, 78.5% opinions', 'Severidad': 'Baja (esperado)',
     'Estrategia': 'Ajuste del análisis: foco en análisis de sentimiento, no extracción artista/canción'},
    {'Fuente': 'Last.fm', 'Campo': 'playcount', 'Naturaleza': 'Distribución muy sesgada (cola pesada)',
     'Frecuencia': 'Artistas top con 4B+ plays vs. media <1B', 'Severidad': 'Alta',
     'Estrategia': 'No se aplica capping (es dato correcto); se usa en rankings y comparaciones relativas'},
    {'Fuente': 'Last.fm', 'Campo': 'mbid', 'Naturaleza': '"unknown" cuando artista no está en MusicBrainz',
     'Frecuencia': 'Variable (~20-40%)', 'Severidad': 'Baja',
     'Estrategia': 'Campo opcional en schema; relleno con "unknown" en silver; no afecta KPIs de negocio'},
    {'Fuente': 'Reddit', 'Campo': 'clean_comment', 'Naturaleza': 'Reducción de longitud post-limpieza',
     'Frecuencia': '100% de filas (esperado)', 'Severidad': 'N/A',
     'Estrategia': 'Pipeline quita HTML, links y puntuación → validado con text_len_mean menor en clean vs raw'},
]

pd.DataFrame(findings).set_index(['Fuente', 'Campo'])

---
## PARTE 5 — Narrativa del Dashboard

> ### ¿Qué historia cuentan los datos?
>
> **La comunidad indie y hip-hop reacciona positivamente a los nuevos lanzamientos, pero raramente habla de canciones específicas.**
>
> **Hallazgo principal:** El 68% de los comentarios analizados expresan sentimiento positivo (VADER compound ≥ 0.05), con un score promedio de +0.62 — un valor alto que indica entusiasmo genuino, no solo positividad superficial. Solo el 14% son negativos.
>
> **Hallazgo secundario (brecha cuantitativa/cualitativa):** Los artistas con más reproducciones en Last.fm (BTS, Taylor Swift, etc.) no son necesariamente los más mencionados en las discusiones de álbumes de Reddit. Las comunidades de r/indieheads y r/hiphopheads focalizan su conversación en artistas de nicho con bases de fans comprometidas.
>
> **Implicación para el usuario funcional (analista/manager):**
> - Un lanzamiento puede generar alta conversación positiva en Reddit con scores de Last.fm modestos → oportunidad de artistas emergentes.
> - La ausencia de menciones estructuradas artista/canción (>95% `unknown`) indica que los usuarios reaccionan al álbum completo, no a tracks individuales → las campañas deberían enfocarse en el concepto del álbum, no en singles.
> - Los keywords más frecuentes con sentimiento positivo (`love`, `great`, `best`, `amazing`) vs. negativos (`disappointing`, `boring`) son las señales más directas de recepción para tomar decisiones de comunicación.

In [ ]:
# ── Resumen ejecutivo final ────────────────────────────────────────────────
print('=' * 60)
print('RESUMEN EJECUTIVO — Gold Layer')
print('=' * 60)

sent_data = df_story[df_story['metric_type'] == 'sentiment_dist']
pos = sent_data[sent_data['dim1'] == 'positive'].iloc[0]
neg = sent_data[sent_data['dim1'] == 'negative'].iloc[0]

print(f'\n📊 Comentarios Reddit analizados: {int(sent_data["record_count"].sum())}')
print(f'   🟢 Positivos: {pos["record_count"]} ({pos["pct"]:.1f}%) · score promedio: {pos["avg_score"]:.3f}')
print(f'   🔴 Negativos: {neg["record_count"]} ({neg["pct"]:.1f}%) · score promedio: {neg["avg_score"]:.3f}')

top_kw = df_story[df_story['metric_type'] == 'top_keyword'].sort_values('record_count', ascending=False)
print(f'\n🔑 Top 5 keywords: {top_kw["dim1"].head(5).tolist()}')

top_artist = df_story[df_story['metric_type'] == 'top_artist_lastfm'].sort_values('avg_score', ascending=False)
print(f'\n🎤 Artista #1 Last.fm por listeners: {top_artist.iloc[0]["dim1"]} ({top_artist.iloc[0]["avg_score"]/1e6:.1f}M listeners)')

top_track = df_story[df_story['metric_type'] == 'top_track_lastfm'].sort_values('record_count', ascending=False)
print(f'🎵 Track #1 Last.fm por plays: {top_track.iloc[0]["dim1"]} — {top_track.iloc[0]["dim2"]} ({top_track.iloc[0]["record_count"]/1e6:.0f}M plays)')

print(f'\n📁 Governance Parquets disponibles: {len(gov_parquets)}')
print(f'📁 Storytelling Parquets disponibles: {len(story_parquets)}')
print('=' * 60)